### RAG-Anything
- 다양한 형식의 문서를 분석하여 그래프 기반의 멀티모달 지식 베이스를 구축하고, 이를 통해 정교한 질의응답을 수행하는 차세대 RAG 프레임워크이다.

1. Multi-modal Content Parsing: 문서 안의 여러 형태 정보를 파싱한다.
> 1) Hierarchical Text Extraction: 문서의 제목, 본문 등 계층적 구조를 유지하며 텍스트 추출
>   - 그냥 텍스트만 뽑는게 아닌 제목 소제목 본문 목록 캡션 비슷한 구조를 최대한 유지하면서 뽑음
> 2) Image Caption & Metadata: 이미지 내 객체를 감지하고 캡션을 생성하며 메타데이터 추출
>   - 문서안의 이미지 설명을 만듦. 예: image3은 샘플링 포인트 수에 따른 MSE변화를 보여준다.
> 3) LaTeX Equation Recognition: 복잡한 수식을 인식하여 기계가 읽을 수 있는 LaTeX 형식으로 변환
>   - pdf 안의 수식이미지를 인식해서 LaTeX로 바꿈. 예: E = mc^2 처럼 기계가 읽도록 변환
> 4) Table Structure Parsing: 표의 행과 열 관계를 유지하며 데이터 구조화

2. Graph-based Multi-modal Knowledge Grounding: 텍스트, 이미지, 표, 수식에서 뽑은 정보를 지식 그래프 형태로 연결한다.
> 1) 텍스트 정보 처리: LLM을 통해 텍스트 내 주요 개념과 이들 간의 관계 파악하고 뽑음.
> 2) 지식 그래프 생성: 추출된 정보를 노드와 엣지로 구성된 그래프 형태로 저장. 그래프는 노드와 엣지로 구성된다.
> 3) VLM/LLM 프로세서: 비전 모델(VLM)을 사용하여 텍스트와 이미지/표 사이의 논리적 연결 고리 생성
>   - 예: 본문에서 Figure4 를 설명함,
>   - Figure4는 MSE변화 그래프임.
>   - 그래프에서 DNN과 LSTM 선이 유사함. 이런 연결을 만든다.
> 4) Merged KG: 여러 문서에서 추출된 개별 그래프들을 하나의 거대한 지식 그래프로 통합
>   - 문서가 여러개 있을때, 각 문서에서 추출한 지식 그래프를 하나로 합침.
>   - 예: 논문 A의 DNN, 논문 B의 DNN, 강의자료의 DNN
>   - 이런 관련 개념들을 연결해서 더 큰 지식베이스를 만든다.

3. Hybrid Query & Retrieval: 질문에 답할때 한가지 검색만 쓰는게 아니라 여러 검색 방식을 섞는다.
> 1) Graph-based Retrieval: 지식 그래프를 탐색하여 질문과 관련된 개념적 관계망 추적
>   - 질문: 크리깅이 DNN 성능 개선에 어떤 영향을 줬나? 라면
>   - 그래프에서: 크리깅 -> 기상 데이터 보간 -> DNN 입력 -> 예측 성능 같은 관계망 찾을 수 있음.
> 2) Embedding-based Retrieval: 벡터 DB를 통해 질문과 유사한 텍스트 청크 및 이미지 정보를 검색
>   - 질문 임베딩 -> 문서 청크 임베딩과 비교 -> 비슷한 문서 조각 검색
> 3) 정보 통합: 그래프 검색 결과와 벡터 검색 결과를 결합하여 LLM에게 전달
>   - 일반 RAG: 벡터 검색 결과만 전달함.
>   - RAG-Anything: 벡터 검색결과 + 그래프 관계 + 이미지/표/수식 해석 결과 전달함

- 표의 방향이 역방향인 경우 분석 능력이 감소할 수 있으며, 이미지보다 텍스트를 우선시하는 경향이 있다.  
<hr>
<div style="text-align:center">
    <h1>
        <a href="https://github.com/HKUDS/RAG-Anything/tree/main">Rag-Anything Git-hub</a>
    </h1>
</div>
<img src="./images/rag_anything_framework.png">

`▼ 실행중인 노트북파일 모두 종료 후 CMD 관리자 권한으로 설치`

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
# %pip install raganything

In [ ]:
# %pip install uv

In [ ]:
# cuda 환경
# %uv pip install -U "mineru[all]"

In [ ]:
# cuda or cpu 환경
# %pip install docling --ignore-installed pydantic-core

In [4]:
# pip install sentence-transformers

In [2]:
# docker run -d --name redis-stack -p 6380:6379 -p 8001:8001 redis/redis-stack-server:latest
# docker exec -it redis-stack redis-cli
from langchain_community.document_loaders import PyMuPDFLoader
from langchain.globals import set_llm_cache
from langchain_community.cache import RedisSemanticCache


from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# OpenAI 대신 로컬 모델 사용
from langchain_huggingface import HuggingFaceEmbeddings

# 1. 캐시용 임베딩 모델 (RAG용 OpenAIEmbeddings와 별개로 로컬 모델 사용 가능)
cache_embeddings = HuggingFaceEmbeddings(model_name="jhgan/ko-sbert-nli")

# 2. 시맨틱 캐시 설정 (Docker로 띄운 Redis Stack 연결)
set_llm_cache(RedisSemanticCache(
    redis_url="redis://localhost:6379",
    embedding=cache_embeddings,
    score_threshold=0.1  # RAG는 답변의 정확도가 중요하므로 임계값을 보수적으로(낮게) 설정
))

# 단계 1: 문서 로드(Load Documents)
FILE_PATH = "./documents/공간 통계 기법을 적용한 기상 데이터 기반 태양광 발전량 예측 모델 연구.pdf"
loader = PyMuPDFLoader(FILE_PATH)
docs = loader.load()
print(f"문서의 페이지수: {len(docs)}")

# 단계 2: 문서 분할(Split Documents)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
split_documents = text_splitter.split_documents(docs)
print(f"분할된 청크의수: {len(split_documents)}")

# 단계 3: 임베딩(Embedding) 생성
# 우리 컴퓨터 내에서 모델을 돌리므로 데이터가 외부로 나가지 않음
embeddings = HuggingFaceEmbeddings(
    model_name="jhgan/ko-sbert-nli", # 한국어 성능이 검증된 오픈소스 모델
    model_kwargs={'device': 'cpu'}    # GPU가 있다면 'cuda'
)

# 단계 4: DB 생성(Create DB) 및 저장
# 벡터스토어 생성
vectorstore = FAISS.from_documents(documents=split_documents, embedding=embeddings)

# 단계 5: 검색기(Retriever) 생성
# 문서에 포함되어 있는 정보를 검색하고 생성한다.
# 1. k=1로 설정: 아주 핵심적인 정보 하나만 딱 본다.
# retriever_1 = vectorstore.as_retriever(search_kwargs={"k": 1})
# 2. k=5로 설정: 주변 맥락까지 폭넓게 읽어서 풍부하게 답변한다.
# retriever_5 = vectorstore.as_retriever(search_kwargs={"k": 5})
# 3. 기본값: k=4
retriever = vectorstore.as_retriever()

# 단계 6: 프롬프트 생성(Create Prompt)
prompt = PromptTemplate.from_template(
    """귀하는 제공된 참고 문헌을 바탕으로 질문에 답하는 정보 분석 전문가입니다. 
    답변 시 반드시 제시된 문맥(Context) 내의 정보만을 활용하십시오. 
    만약 주어진 자료만으로 답변이 어렵다면, 추측하지 말고 
    '제공된 정보로는 확인이 불가능하다'고 명확히 밝히십시오. 
    모든 응답은 한국어로 작성합니다.

#Context: 
{context}

#Question:
{question}

#Answer:"""
)

# 단계 7: 언어모델(LLM) 생성
llm = ChatOpenAI(model_name="gpt-4.1-mini", temperature=0)

# 단계 8: 체인(Chain) 생성
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 체인 실행(Run Chain)
# 문서에 대한 질의를 입력하고, 답변을 출력합니다.
# question = """
# 병렬 처리 기반 크리깅을 통해 산출한 발전소 위치의 추정 기상 데이터를 입력으로 적용한 경우 
# DNN의 2시간 후 예측에서 각 평가지표 별로 얼마나 개선되었나?
# """
question = """
MSE variation with respect to the number of sampling points 그래프에서 
어떤 모델끼리 비슷한가(겹쳐있는가)? 
"""

response = chain.invoke(question)
print(response)

ImportError: Could not import sentence_transformers python package. Please install it with `pip install sentence-transformers`.

In [3]:
import asyncio
import os
from functools import partial
from dotenv import load_dotenv
import numpy as np

from raganything import RAGAnything, RAGAnythingConfig
from lightrag.llm.openai import openai_complete_if_cache
from lightrag.utils import EmbeddingFunc
from langchain_huggingface import HuggingFaceEmbeddings

# CPU 사용 강제 설정
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

# .env 파일 로드
load_dotenv()

async def main():
    # 환경 변수에서 API 키 가져오기
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        print("에러: .env 파일에 OPENAI_API_KEY가 설정되어 있지 않습니다.")
        return

    # 1. RAGAnything 설정
    config = RAGAnythingConfig(
        # rag_storage
        # RAG 시스템의 영구 기억장치
        # 문서 분석의 결과를 저장하기 때문에 동일한 질문을 할 때 문서를 다시 읽을 필요 없다.
        working_dir="./rag_storage",
        # parser="mineru", 
        parser="docling",
        parse_method="auto",
        enable_image_processing=True,
        enable_table_processing=True,
        enable_equation_processing=True,
    )

    # 2. LLM 모델 함수 (gpt-5.4-nano)
    def llm_model_func(prompt, system_prompt=None, history_messages=[], **kwargs):
        return openai_complete_if_cache(
            "gpt-5.4-nano",
            prompt,
            system_prompt=system_prompt,
            history_messages=history_messages,
            api_key=api_key,
            **kwargs,
        )

    # 3. 비전 모델 함수 (gpt-5.4-mini)
    def vision_model_func(prompt, system_prompt=None, history_messages=[], image_data=None, messages=None, **kwargs):
        if messages:
            return openai_complete_if_cache("gpt-5.4-mini", "", messages=messages, api_key=api_key, **kwargs)
        elif image_data:
            return openai_complete_if_cache(
                "gpt-5.4-mini", "",
                messages=[
                    {"role": "system", "content": system_prompt} if system_prompt else None,
                    {
                        "role": "user",
                        "content": [
                            {"type": "text", "text": prompt},
                            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_data}"}},
                        ],
                    }
                ],
                api_key=api_key,
                **kwargs,
            )
        else:
            return llm_model_func(prompt, system_prompt, history_messages, **kwargs)

    # 4. 로컬 HuggingFace 임베딩 설정 (ko-sbert-nli)
    hf_embeddings = HuggingFaceEmbeddings(model_name="jhgan/ko-sbert-nli")
    
    async def local_embedding_func_wrapper(texts):
        # 리스트 형태로 임베딩 추출
        embeddings = hf_embeddings.embed_documents(texts)
        # 내부적으로 size함수를 사용하기 때문에 ndarray로 변환
        return np.array(embeddings)

    embedding_func = EmbeddingFunc(
        embedding_dim=768, # ko-sbert 모델의 출력 차원
        max_token_size=512,
        func=local_embedding_func_wrapper
    )

    # 5. RAGAnything 초기화
    rag = RAGAnything(
        config=config,
        llm_model_func=llm_model_func,
        vision_model_func=vision_model_func,
        embedding_func=embedding_func,
    )

    # 6. 문서 처리
    FILE_PATH = "./documents/korea_trade_flow_report.pdf"
    
    if not os.path.exists(FILE_PATH):
        print(f"파일을 찾을 수 없습니다: {FILE_PATH}")
        return

    print("문서 분석 및 멀티모달 지식 그래프 구축 시작...")
    await rag.process_document_complete(
        file_path=FILE_PATH,
        # PDF 내에서 추출된 표(Table), 그림(Figure), 차트 이미지 파일들이 저장된다.
        output_dir="./rag_output",
        parse_method="auto"
    )

    # 7. 질의 실행
    question = """
    MSE variation with respect to the number of sampling points 그래프에서 
    어떤 모델끼리 비슷한가(겹쳐있는가)?
    """

    print(f"\n[질문]: {question}")
    
    # 텍스트 쿼리 실행 (aquery 사용)
    result = await rag.aquery(
        question,
        mode="hybrid"
    )
    print(f"\n[분석 결과]:\n{result}")

try:
    # 파이썬 환경
    # asyncio.run(main())
    
    # 주피터 노트북 환경
    await main()
    
except Exception as e:
    print(f"실행 중 오류 발생: {e}")

실행 중 오류 발생: Could not import sentence_transformers python package. Please install it with `pip install sentence-transformers`.
